In [ ]:
# Fabric Notebook: NB_30_SILVER_POI_ENTITY_ENRICHMENT
# READ/WRITE: Silver only
# Default Lakehouse: lh_vv_silver (REQUIRED)
# Builds Silver poi_enrichment lifecycle and poi_entity SCD2.
#
# IMPORTANT:
# - NB30 MUST be attached to/default on Lakehouse: lh_vv_silver, schema dbo.
# - NB00 must invoke NB30 with useRootDefaultLakehouse=True only when NB00 itself
#   is running with lh_vv_silver as the root/default Lakehouse.
# - In this Fabric Spark session, Silver tables resolve correctly as SINGLE-PART
#   names. Do NOT use <table> inside NB30 SQL because that
#   triggers [REQUIRES_SINGLE_PART_NAMESPACE].
#
# Expected physical tables:
#   _stage_poi_language
#   poi_entity
#   poi_enrichment
#   ...
#
# SQL inside this notebook intentionally uses:
#   _stage_poi_language
#   poi_entity
#   poi_enrichment
#   ...

run_id = globals().get("run_id", "DEV_RUN")

STAGE              = "_stage_poi_language"
ENR                = "poi_enrichment"
ENTITY             = "poi_entity"
POI_REVIEW         = "poi_review"
POI_MEDIA          = "poi_media"
POI_CONTENT        = "poi_content"
POI_AMENITY_SOURCE = "poi_amenity_source"
POI_ADDRESS        = "poi_address"
POI_CONTACT        = "poi_contact"
POI_OPENING_HOURS  = "poi_opening_hours"
POI_RATING         = "poi_rating"
POI_POLICY         = "poi_policy"

print(f"NB30 run_id = {run_id}")

# Diagnostic: confirm NB30 is really executing in the expected Silver context.
_ctx = spark.sql("""
SELECT current_catalog() AS current_catalog,
       current_database() AS current_database
""").first()

print(f"current_catalog  = {_ctx['current_catalog']}")
print(f"current_database = {_ctx['current_database']}")

# Fail fast for tables that must already exist before NB30.
# poi_enrichment is intentionally excluded because NB30 owns its DDL.
_required_existing = [
    STAGE,
    ENTITY,
    POI_REVIEW,
    POI_MEDIA,
    POI_CONTENT,
    POI_AMENITY_SOURCE,
    POI_ADDRESS,
    POI_CONTACT,
    POI_OPENING_HOURS,
    POI_RATING,
    POI_POLICY,
]

_missing = []
for _tbl in _required_existing:
    try:
        spark.sql(f"SELECT 1 FROM {_tbl} LIMIT 1").collect()
    except Exception as _e:
        _missing.append(f"{_tbl}: {_e}")

if _missing:
    raise RuntimeError(
        "NB30 missing/unreadable required Silver tables in current default "
        "Lakehouse. Expected \n"
        + "\n".join(_missing)
    )

print("NB30 Silver preflight = OK")

# -----------------------------------------------------------------------------
# 1. Silver poi_enrichment DDL
# -----------------------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ENR} (
  enrichment_id STRING,
  poi_id STRING,
  poi_group_id STRING,
  lang STRING,
  source_enrichment_poi_id STRING,
  source_id STRING,
  model_name STRING,
  ref_business_sector STRING,
  ref_type STRING,
  description STRING,
  time_to_spend_minutes INT,
  best_time_type STRING,
  best_time_from_month INT,
  best_time_to_month INT,
  best_time_value STRING,
  cost_vnd BIGINT,
  cost_method STRING,
  cost_confidence STRING,
  experiences_json STRING,
  review_summary STRING,
  reasoning_short STRING,
  confidence STRING,
  needs_manual_review BOOLEAN,
  poi_name STRING,
  short_address STRING,
  district STRING,
  story STRING,
  usp_highlights_json STRING,
  local_tips_or_neighborhood STRING,
  cancellation_policy STRING,
  reviews_json STRING,
  enriched_at TIMESTAMP,
  is_current BOOLEAN,
  source_event_id STRING,
  source_crawled_at TIMESTAMP,
  run_id STRING,
  _effective_from TIMESTAMP,
  _effective_to TIMESTAMP,
  _created_at TIMESTAMP,
  _updated_at TIMESTAMP
) USING DELTA
""")

# Dedup incoming enrichment by enrichment_id, latest crawled_at; skip rows without enrichment.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_incoming_enrichment AS
SELECT * FROM (
 SELECT
   enrichment_id,poi_id,poi_group_id,lang,source_enrichment_poi_id,source_id,
   get_json_object(localized_enrichment_json,'$.modelName') model_name,
   get_json_object(localized_enrichment_json,'$.refBusinessSector') ref_business_sector,
   get_json_object(localized_enrichment_json,'$.refType') ref_type,
   get_json_object(localized_enrichment_json,'$.description') description,
   CAST(get_json_object(localized_enrichment_json,'$.timeToSpendMinutes') AS INT) time_to_spend_minutes,
   get_json_object(localized_enrichment_json,'$.bestTime.type') best_time_type,
   CAST(get_json_object(localized_enrichment_json,'$.bestTime.fromMonth') AS INT) best_time_from_month,
   CAST(get_json_object(localized_enrichment_json,'$.bestTime.toMonth') AS INT) best_time_to_month,
   get_json_object(localized_enrichment_json,'$.bestTime.value') best_time_value,
   CAST(get_json_object(localized_enrichment_json,'$.costVnd') AS BIGINT) cost_vnd,
   get_json_object(localized_enrichment_json,'$.costMethod') cost_method,
   get_json_object(localized_enrichment_json,'$.costConfidence') cost_confidence,
   get_json_object(localized_enrichment_json,'$.experiences') experiences_json,
   get_json_object(localized_enrichment_json,'$.reviewSummary') review_summary,
   get_json_object(localized_enrichment_json,'$.reasoningShort') reasoning_short,
   get_json_object(localized_enrichment_json,'$.confidence') confidence,
   CAST(get_json_object(localized_enrichment_json,'$.needsManualReview') AS BOOLEAN) needs_manual_review,
   get_json_object(localized_enrichment_json,'$.poiName') poi_name,
   get_json_object(localized_enrichment_json,'$.shortAddress') short_address,
   get_json_object(localized_enrichment_json,'$.district') district,
   get_json_object(localized_enrichment_json,'$.story') story,
   get_json_object(localized_enrichment_json,'$.uspHighlights') usp_highlights_json,
   get_json_object(localized_enrichment_json,'$.localTipsOrNeighborhood') local_tips_or_neighborhood,
   get_json_object(localized_enrichment_json,'$.cancellationPolicy') cancellation_policy,
   get_json_object(localized_enrichment_json,'$.reviews') reviews_json,
   CAST(get_json_object(localized_enrichment_json,'$.enrichedAt') AS TIMESTAMP) enriched_at,
   event_id source_event_id,crawled_at source_crawled_at,'{run_id}' run_id,
   ROW_NUMBER() OVER(PARTITION BY enrichment_id ORDER BY crawled_at DESC,event_id DESC) rn
 FROM {STAGE}
 WHERE run_id='{run_id}' AND localized_enrichment_json IS NOT NULL AND enrichment_id IS NOT NULL
) WHERE rn=1
""")

# -----------------------------------------------------------------------------
# 1B. poi_enrichment lifecycle - idempotent + exactly one current per poi_id
# -----------------------------------------------------------------------------
#
# Guarantees:
#   - enrichment_id inserted at most once
#   - at most one is_current=true per poi_id
#   - multiple new versions in the same run are preserved as history
#   - latest version per poi_id becomes current
#   - historical duplicate-current rows are self-healed

# 1B.1 Self-heal existing duplicate current rows.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_existing_current_ranked AS
SELECT
    enrichment_id,
    poi_id,
    ROW_NUMBER() OVER (
        PARTITION BY poi_id
        ORDER BY
            enriched_at DESC NULLS LAST,
            source_crawled_at DESC NULLS LAST,
            _created_at DESC NULLS LAST,
            enrichment_id DESC
    ) AS rn
FROM {ENR}
WHERE is_current = true
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_existing_current_to_close AS
SELECT enrichment_id
FROM v2_existing_current_ranked
WHERE rn > 1
""")

spark.sql(f"""
MERGE INTO {ENR} tgt
USING v2_existing_current_to_close src
ON tgt.enrichment_id = src.enrichment_id
WHEN MATCHED THEN UPDATE SET
    tgt.is_current = false,
    tgt._effective_to = COALESCE(tgt._effective_to, current_timestamp()),
    tgt._updated_at = current_timestamp()
""")

# 1B.2 Keep only unseen enrichment versions.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_new_enrichment AS
SELECT i.*
FROM v2_incoming_enrichment i
LEFT ANTI JOIN {ENR} existing
  ON existing.enrichment_id = i.enrichment_id
""")

# 1B.3 Rank all new enrichment versions per poi_id.
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_new_enrichment_ranked AS
SELECT
    i.*,
    ROW_NUMBER() OVER (
        PARTITION BY poi_id
        ORDER BY
            enriched_at DESC NULLS LAST,
            source_crawled_at DESC NULLS LAST,
            enrichment_id DESC
    ) AS poi_rank
FROM v2_new_enrichment i
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_new_enrichment_winner AS
SELECT *
FROM v2_new_enrichment_ranked
WHERE poi_rank = 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_new_enrichment_history AS
SELECT *
FROM v2_new_enrichment_ranked
WHERE poi_rank > 1
""")

# 1B.4 Close current rows for POIs that receive a new winner.
spark.sql(f"""
MERGE INTO {ENR} tgt
USING (
    SELECT DISTINCT poi_id, enrichment_id, enriched_at
    FROM v2_new_enrichment_winner
) src
ON tgt.poi_id = src.poi_id
 AND tgt.is_current = true
 AND tgt.enrichment_id <> src.enrichment_id
WHEN MATCHED THEN UPDATE SET
    tgt.is_current = false,
    tgt._effective_to = COALESCE(src.enriched_at, current_timestamp()),
    tgt._updated_at = current_timestamp()
""")

# 1B.5 Insert non-winning new versions as history.
spark.sql(f"""
INSERT INTO {ENR}
SELECT
    i.enrichment_id,
    i.poi_id,
    i.poi_group_id,
    i.lang,
    i.source_enrichment_poi_id,
    i.source_id,
    i.model_name,
    i.ref_business_sector,
    i.ref_type,
    i.description,
    i.time_to_spend_minutes,
    i.best_time_type,
    i.best_time_from_month,
    i.best_time_to_month,
    i.best_time_value,
    i.cost_vnd,
    i.cost_method,
    i.cost_confidence,
    i.experiences_json,
    i.review_summary,
    i.reasoning_short,
    i.confidence,
    i.needs_manual_review,
    i.poi_name,
    i.short_address,
    i.district,
    i.story,
    i.usp_highlights_json,
    i.local_tips_or_neighborhood,
    i.cancellation_policy,
    i.reviews_json,
    i.enriched_at,
    false,
    i.source_event_id,
    i.source_crawled_at,
    i.run_id,
    COALESCE(i.enriched_at, current_timestamp()),
    COALESCE(
        LEAD(i.enriched_at) OVER (
            PARTITION BY i.poi_id
            ORDER BY
                i.enriched_at ASC NULLS FIRST,
                i.source_crawled_at ASC NULLS FIRST,
                i.enrichment_id ASC
        ),
        w.enriched_at,
        current_timestamp()
    ),
    current_timestamp(),
    current_timestamp()
FROM v2_new_enrichment_history i
LEFT JOIN v2_new_enrichment_winner w
  ON w.poi_id = i.poi_id
""")

# 1B.6 Insert exactly one current winner per poi_id.
spark.sql(f"""
INSERT INTO {ENR}
SELECT
    i.enrichment_id,
    i.poi_id,
    i.poi_group_id,
    i.lang,
    i.source_enrichment_poi_id,
    i.source_id,
    i.model_name,
    i.ref_business_sector,
    i.ref_type,
    i.description,
    i.time_to_spend_minutes,
    i.best_time_type,
    i.best_time_from_month,
    i.best_time_to_month,
    i.best_time_value,
    i.cost_vnd,
    i.cost_method,
    i.cost_confidence,
    i.experiences_json,
    i.review_summary,
    i.reasoning_short,
    i.confidence,
    i.needs_manual_review,
    i.poi_name,
    i.short_address,
    i.district,
    i.story,
    i.usp_highlights_json,
    i.local_tips_or_neighborhood,
    i.cancellation_policy,
    i.reviews_json,
    i.enriched_at,
    true,
    i.source_event_id,
    i.source_crawled_at,
    i.run_id,
    COALESCE(i.enriched_at, current_timestamp()),
    NULL,
    current_timestamp(),
    current_timestamp()
FROM v2_new_enrichment_winner i
""")

# 1B.7 Final safety repair for partially failed historical runs.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_final_current_ranked AS
SELECT
    enrichment_id,
    poi_id,
    ROW_NUMBER() OVER (
        PARTITION BY poi_id
        ORDER BY
            enriched_at DESC NULLS LAST,
            source_crawled_at DESC NULLS LAST,
            _created_at DESC NULLS LAST,
            enrichment_id DESC
    ) AS rn
FROM {ENR}
WHERE is_current = true
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_final_current_to_close AS
SELECT enrichment_id
FROM v2_final_current_ranked
WHERE rn > 1
""")

spark.sql(f"""
MERGE INTO {ENR} tgt
USING v2_final_current_to_close src
ON tgt.enrichment_id = src.enrichment_id
WHEN MATCHED THEN UPDATE SET
    tgt.is_current = false,
    tgt._effective_to = COALESCE(tgt._effective_to, current_timestamp()),
    tgt._updated_at = current_timestamp()
""")

# 2. poi_entity SCD2 - impacted POIs only; broader tracked hash than V1.
# -----------------------------------------------------------------------------
try:
    spark.sql(f"ALTER TABLE {ENTITY} ADD COLUMNS (poi_group_id STRING)")
except Exception:
    pass

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_entity_incoming AS
SELECT
  s.poi_id,s.poi_group_id,s.lang,
  s.first_seen_at,s.last_seen_at,s.source_name canonical_source,
  s.poi_name,
  s.poi_name_normalized normalized_name,
  COALESCE(s.business_sector,'other') business_sector,
  s.business_category,
  from_json(s.types_json,'ARRAY<STRING>') subcategory_tags,
  COALESCE(s.operating_status,'operational') operating_status,
  s.slug,s.partner_id,s.source_id,s.event_id,s.event_id _batch_id,
  sha2(concat_ws('|',
    s.poi_id,COALESCE(s.poi_name,''),COALESCE(s.poi_name_normalized,''),
    COALESCE(s.business_sector,''),COALESCE(s.business_category,''),COALESCE(s.types_json,''),
    COALESCE(s.operating_status,''),COALESCE(s.slug,''),COALESCE(s.partner_id,''),
    COALESCE(s.source_name,''),COALESCE(s.source_id,'')
  ),256) entity_version_id
FROM {STAGE} s
WHERE s.run_id='{run_id}' AND s.rn_by_poi=1
""")

# Compute quality scores but do NOT include them in entity_version_id.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_entity_enriched AS
WITH flags AS (
 SELECT poi_id,
  MAX(CASE WHEN src='review' THEN 1 ELSE 0 END) has_review,
  MAX(CASE WHEN src='media' THEN 1 ELSE 0 END) has_media,
  MAX(CASE WHEN src='content' THEN 1 ELSE 0 END) has_content,
  MAX(CASE WHEN src='amenity' THEN 1 ELSE 0 END) has_amenity
 FROM (
   SELECT poi_id,'review' src FROM {POI_REVIEW} GROUP BY poi_id
   UNION ALL SELECT poi_id,'media' FROM {POI_MEDIA} GROUP BY poi_id
   UNION ALL SELECT poi_id,'content' FROM {POI_CONTENT} GROUP BY poi_id
   UNION ALL SELECT poi_id,'amenity' FROM {POI_AMENITY_SOURCE} GROUP BY poi_id
 ) x GROUP BY poi_id
)
SELECT i.*,
 (CASE WHEN a.poi_id IS NOT NULL THEN 15 ELSE 0 END+
  CASE WHEN c.poi_id IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN oh.poi_id IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN r.poi_id IS NOT NULL THEN 15 ELSE 0 END+
  CASE WHEN f.has_review=1 THEN 10 ELSE 0 END+
  CASE WHEN f.has_media=1 THEN 15 ELSE 0 END+
  CASE WHEN f.has_content=1 THEN 15 ELSE 0 END+
  CASE WHEN pl.poi_id IS NOT NULL THEN 5 ELSE 0 END+
  CASE WHEN f.has_amenity=1 THEN 5 ELSE 0 END) enrichment_score,
 (CASE WHEN a.full_address IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN a.lat IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN c.phone IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN c.website_url IS NOT NULL THEN 5 ELSE 0 END+
  CASE WHEN oh.poi_id IS NOT NULL THEN 10 ELSE 0 END+
  CASE WHEN r.overall IS NOT NULL THEN 15 ELSE 0 END+
  CASE WHEN f.has_media=1 THEN 15 ELSE 0 END+
  CASE WHEN pl.policy_json IS NOT NULL THEN 5 ELSE 0 END+
  CASE WHEN f.has_amenity=1 THEN 5 ELSE 0 END) data_completeness
FROM v2_entity_incoming i
LEFT JOIN flags f ON f.poi_id=i.poi_id
LEFT JOIN {POI_ADDRESS} a ON a.poi_id=i.poi_id
LEFT JOIN {POI_CONTACT} c ON c.poi_id=i.poi_id
LEFT JOIN {POI_OPENING_HOURS} oh ON oh.poi_id=i.poi_id
LEFT JOIN {POI_RATING} r ON r.poi_id=i.poi_id
LEFT JOIN {POI_POLICY} pl ON pl.poi_id=i.poi_id
""")

spark.sql(f"""
MERGE INTO {ENTITY} tgt
USING (SELECT poi_id,entity_version_id FROM v2_entity_enriched) src
ON tgt.poi_id=src.poi_id AND tgt._is_current=true AND tgt.entity_version_id<>src.entity_version_id
WHEN MATCHED THEN UPDATE SET tgt._effective_to=current_timestamp(),tgt._is_current=false,tgt._change_reason='entity_fields_changed',tgt._updated_at=current_timestamp()
""")

# Materialize rows to insert OUTSIDE the INSERT statement so the fully-qualified
# ENTITY target is not read again inside INSERT SELECT.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_entity_to_insert AS
SELECT e.*
FROM v2_entity_enriched e
LEFT ANTI JOIN {ENTITY} x
  ON x.poi_id = e.poi_id
 AND x._is_current = true
""")

spark.sql(f"""
INSERT INTO {ENTITY} (
 poi_id,poi_group_id,entity_version_id,lang,poi_name,normalized_name,business_sector,business_category,subcategory_tags,
 brand_name,chain_id,slug,operating_status,lifecycle_state,lifecycle_state_reason,is_active,is_verified,verified_at,verified_by,
 partner_id,canonical_source,source_id,event_id,merge_target_poi_id,enrichment_score,data_completeness,
 source_first_seen_at,source_last_seen_at,_effective_from,_effective_to,_is_current,_change_reason,_batch_id,_created_at,_updated_at
)
SELECT
 e.poi_id,e.poi_group_id,e.entity_version_id,e.lang,e.poi_name,e.normalized_name,e.business_sector,e.business_category,e.subcategory_tags,
 NULL,NULL,e.slug,e.operating_status,'published',NULL,true,false,NULL,NULL,e.partner_id,e.canonical_source,e.source_id,e.event_id,NULL,
 e.enrichment_score,e.data_completeness,e.first_seen_at,e.last_seen_at,
 current_timestamp(),NULL,true,NULL,e._batch_id,current_timestamp(),current_timestamp()
FROM v2_entity_to_insert e
""")

# Operational counters
incoming_enrichment_count = spark.sql("SELECT COUNT(*) c FROM v2_incoming_enrichment").first()["c"]
new_enrichment_count = spark.sql("SELECT COUNT(*) c FROM v2_new_enrichment").first()["c"]
new_enrichment_winner_count = spark.sql("SELECT COUNT(*) c FROM v2_new_enrichment_winner").first()["c"]
new_enrichment_history_count = spark.sql("SELECT COUNT(*) c FROM v2_new_enrichment_history").first()["c"]
entity_incoming_count = spark.sql("SELECT COUNT(*) c FROM v2_entity_enriched").first()["c"]
entity_insert_count = spark.sql("SELECT COUNT(*) c FROM v2_entity_to_insert").first()["c"]

print(f"incoming_enrichment_count = {incoming_enrichment_count}")
print(f"new_enrichment_count      = {new_enrichment_count}")
print(f"new_enrichment_winner_count = {new_enrichment_winner_count}")
print(f"new_enrichment_history_count = {new_enrichment_history_count}")
print(f"entity_incoming_count     = {entity_incoming_count}")
print(f"entity_insert_count       = {entity_insert_count}")

# Invariants
bad_enr = spark.sql(
    f"SELECT poi_id FROM {ENR} "
    "WHERE is_current=true "
    "GROUP BY poi_id HAVING COUNT(*)>1 LIMIT 1"
).count()

bad_enr_id = spark.sql(
    f"SELECT enrichment_id FROM {ENR} "
    "GROUP BY enrichment_id HAVING COUNT(*)>1 LIMIT 1"
).count()

bad_entity = spark.sql(
    f"SELECT poi_id FROM {ENTITY} "
    "WHERE _is_current=true "
    "GROUP BY poi_id HAVING COUNT(*)>1 LIMIT 1"
).count()

if bad_enr or bad_enr_id or bad_entity:
    raise Exception(
        "CURRENT_ROW_INVARIANT_FAILED "
        f"enrichment_current={bad_enr} "
        f"enrichment_id_duplicate={bad_enr_id} "
        f"entity_current={bad_entity}"
    )

notebookutils.notebook.exit("SUCCESS")
